In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import numpy as np
import os, getpass, random, copy, torch, gc
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
df_path = os.path.join(script_dir, "Datasets","Concrete Strength.csv")
df = pd.read_csv(df_path)
df = df.dropna()

x = df.drop(columns="concrete_strength_mpa").values
y = df["concrete_strength_mpa"].values.reshape(-1,1)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

x = x_scaler.fit_transform(x)
y = y_scaler.fit_transform(y)

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.245, random_state=38)
x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

user_account = getpass.getuser()
predictions_bar_chart = os.path.join(script_dir, "Concrete Strength Predictor Bar Chart.png")

class Regressor(nn.Module):
    def __init__(self, input_dim=x_train.shape[1]):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=19),
            nn.LeakyReLU(),
            nn.BatchNorm1d(19),

            nn.Linear(in_features=19, out_features=27),
            nn.LeakyReLU(),
            nn.BatchNorm1d(27),

            nn.Linear(in_features=27, out_features=41),
            nn.LeakyReLU(),
            nn.BatchNorm1d(41),

            nn.Linear(in_features=41, out_features=56),
            nn.LeakyReLU(),
            nn.BatchNorm1d(56),

            nn.Linear(in_features=56, out_features=68),
            nn.LeakyReLU(),
            nn.BatchNorm1d(68),
            nn.Dropout(p=0.6),

            nn.Linear(in_features=68, out_features=74),
            nn.LeakyReLU(),
            nn.BatchNorm1d(74), 
            nn.Dropout(p=0.8),
            
            nn.Linear(in_features=74, out_features=82),
            nn.LeakyReLU(),
            nn.BatchNorm1d(82), 
            nn.Dropout(p=0.3),

            nn.Linear(in_features=82, out_features=62),
            nn.LeakyReLU(),
            nn.BatchNorm1d(62), 

            nn.Linear(in_features=62, out_features=35),
            nn.LeakyReLU(),
            nn.BatchNorm1d(35), 

            nn.Linear(in_features=35, out_features=11),
            nn.LeakyReLU(),
            nn.BatchNorm1d(11),   

            nn.Linear(in_features=11, out_features=1),                       
        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

losses = []
def run_grid_search(rates_list, epoch_counts_list, seed_val):
    best_state = None
    loss_fn = nn.L1Loss()

    print(f"\n== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<15} | {'Val Loss':<15}")
    print(f"="*65) 

    for lr in rates_list:
        for epoch_count in epoch_counts_list:
            torch.manual_seed(int(seed_val))
            model = Regressor().to(device)

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            for epoch in tqdm(range(epoch_count)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_loss = loss_fn(model(x_test), y_test)

            print(f"{lr:<15} | {epoch_count:<15} | {loss:<15}")
            losses.append(test_loss)

            del model, optimizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            best_state = copy.deepcopy(model.state_dict())    
    print(f"\n● Best Loss = {min(losses):.2f}")
    print(f"● Worst Loss = {max(losses):.2f}")

    return best_state

epoch_count_list = [950,1020,780,810]
rates_list = [0.01, 0.1, 0.001, 0.0001]
seed_val = float(input("-- Enter a Value for a Seed : ").strip())

best_state = run_grid_search(epoch_counts_list=epoch_count_list, rates_list=rates_list, seed_val=seed_val)
torch.save(best_state, f"Concrete Strength Predictor Parameters")

fresh_model = Regressor().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

inputs = []
predictions = []

while True:
    user_name = input("-- Enter your Name ( or type 'exit' to quit ) : ").strip()

    if user_name.lower() == "exit":
        print(f"Exiting Interactive Loop ...")
        break

    cement_kg = float(input("-- How much cement do you have in Kg : ").strip())
    water_kg = float(input("-- How much water do you have in Kg : ").strip())
    sand_kg = float(input("-- How much sand do you have in Kg : ").strip())
    gravel_kg = float(input('-- How much gravel do you have in Kg : ').strip())
    age_days = float(input("-- How old is the Concrete in Days : ").strip())

    raw_array = np.array([cement_kg, water_kg, sand_kg, gravel_kg, age_days])
    scaled_array = x_scaler.transform(raw_array)
    scaled_tensor = torch.tensor(scaled_array, dtype=torch.float32).to(device)

    with torch.inference_mode():
        preds = fresh_model(scaled_tensor).cpu().numpy()
    untransformed = y_scaler.inverse_transform(preds).item()
    print(f"Predicted Concrete Strength for you, {user_name.title()} = {round(untransformed)} mpa")

    inputs.append(f"{user_name.title()} ( {cement_kg} Kg of Cement )")
    predictions.append(round(untransformed))

try:
    if inputs and predictions:
        plt.figure(figsize=(15,4))
        colors = ["red","blue","orange","yellow","purple"]
        bar_colors = [random.choice(colors) for _ in range(len(predictions))]
        plt.bar(inputs, predictions, color=bar_colors, edgecolor="black")
        plt.title("Predictions Bar Chart = Concrete Strength Predictor")
        plt.xlabel("People | Cement Amount ( Kg )")
        plt.ylabel("Predicted Concrete Strength Magnitudes")
        plt.grid()
        plt.legend(prop={"size":14,"weight":"bold"})
        plt.savefig(predictions_bar_chart)
        plt.close()
        print(f"Successfully Saved the Graph")
    else:
        print(f"Failed to Save Graph = No Inputs or Predictions")    
except BaseException as e:
    print(f"Failed to Save Graph = {e}")        

try:
    one_drive = input("-- Do you use One Drive or Not : ").strip().title()
    if one_drive == "Yes":
        desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Concrete Strength Predictor Details.txt"
    elif one_drive == "No":
        desktop_path = f"C:/Users/{user_account}/Desktop/Concrete Strength Predictor Details.txt"
except BaseException as e:
    print(f"Invalid Input. Error has occured = {e}")

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
        file.write("== Concrete Strength Predictor Details ")
        file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.2f}")
        file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.2f}")
        file.write(f"\n● Best Loss = {min(losses):.2f}")
        file.write(f"● Worst Loss = {max(losses):.2f}")
    print(f"Successfully Saved your Model Details")
except BaseException as e:
    print(f"Failed to Save your Network Details = {e}")        

_IncompleteInputError: incomplete input (3867061379.py, line 195)